# Unit 8 / Chapter 8: Quantum Reinforcement Learning

> **Main Learning Objective:** Understand reinforcement learning as the third major branch of AI, see where quantum methods can plausibly speed up parts of it, and build a tiny quantum policy gradient agent from scratch.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 8.1 | RL in one page | The framework behind AlphaGo, ChatGPT RLHF, robotics |
| 8.2 | Where quantum can help RL | Value estimation, policy representation, sampling |
| 8.3 | Quantum policy gradients | A parameterized quantum circuit as a policy |
| 8.4 | Grover for RL search | Speeding up action selection in large action spaces |

By the end of this unit you should be able to:

1. State the reinforcement learning problem in one sentence.
2. Name the three main components: policy, value function, and environment.
3. Explain what a quantum policy is and why it uses a parameterized quantum circuit.
4. Describe one way Grover's algorithm can accelerate action selection.
5. Identify at least one AI application (robotics, games, LLMs) where quantum RL could plausibly help.

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 8**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 8
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 8.1: Reinforcement Learning in One Page

Reinforcement learning (RL) is the third major branch of machine learning, after supervised and unsupervised learning. In RL:

- An **agent** interacts with an **environment**.
- At each step it observes a **state** s, picks an **action** a, and receives a **reward** r.
- The goal is to maximize the sum of rewards over time.

The **policy** pi(a given s) is the function that picks actions. The **value function** V(s) estimates how much reward you will collect starting from state s.

RL is behind:

| Application | RL role |
|---|---|
| AlphaGo, chess engines | Learn to play games at superhuman level |
| Robot control | Learn to walk, grasp, drive |
| ChatGPT RLHF | Fine tune LLMs from human preferences |
| Trading | Optimize buy/sell decisions |
| Recommender systems | Optimize long term engagement |

In [ ]:
# Toy RL environment: a 5 state chain, moving right earns 1 reward, left earns 0
# Simple value iteration to see the RL loop
N = 5; gamma = 0.9
V = np.zeros(N)
for step in range(100):
    V_new = V.copy()
    for s in range(N):
        # right action
        s_right = min(s+1, N-1)
        v_right = 1.0 + gamma * V[s_right] if s < N-1 else 0.0
        # left action
        s_left = max(s-1, 0)
        v_left  = 0.0 + gamma * V[s_left]
        V_new[s] = max(v_right, v_left)
    V = V_new

plt.bar(range(N), V, color='#5B2C91')
plt.xlabel('state s'); plt.ylabel('V(s)')
plt.title('Learned value function on a 5-state chain (higher = more future reward)')
plt.tight_layout(); plt.show()
print("V(s) tells the agent which states are worth being in.")

---
# Section 8.2: Where Quantum Can Help RL

Reinforcement learning has three big computational bottlenecks:

1. **Estimating value functions** in huge state spaces (billions of states in Go, uncountable in continuous robotics).
2. **Representing the policy** compactly. Neural networks help, but the parameter count explodes.
3. **Sampling** trajectories through the environment.

Quantum can plausibly help with all three:

- **Value estimation:** Quantum linear systems (Unit 6) can accelerate Bellman equation solves for structured problems.
- **Policy representation:** A parameterized quantum circuit is a compact function approximator (Unit 3, 4).
- **Sampling:** Quantum sampling (Unit 2) is a natural fit for exploring rollouts in a stochastic policy.

The current honest state of the field is that all of these are research topics, not production-ready tools. Most quantum RL results are toy demonstrations.

In [ ]:
# A quantum policy is just a PQC where measurement outcomes are actions.
# We'll build the simplest possible one on 1 qubit -> 2 actions.

def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]])
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]])

def quantum_policy(state, theta):
    '''Encode state, apply Ry(theta), measure. Returns P(action=0), P(action=1).'''
    psi = np.array([1.0, 0.0], dtype=complex)
    psi = Ry(state) @ psi   # state-dependent rotation
    psi = Ry(theta) @ psi   # parameterized rotation
    p0 = abs(psi[0])**2
    return p0, 1 - p0

# Sweep theta to see how action probabilities change
states = np.linspace(0, np.pi, 40)
thetas = [0.0, np.pi/2, np.pi]
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, th in zip(axes, thetas):
    p0s = [quantum_policy(s, th)[0] for s in states]
    ax.plot(states, p0s, color='#5B2C91', lw=2)
    ax.set_title(f'theta = {th:.2f}')
    ax.set_xlabel('state'); ax.set_ylim(0, 1)
axes[0].set_ylabel('P(action = 0)')
plt.tight_layout(); plt.show()
print("Different theta values give different policies. Training theta gives you a learned policy.")

---
# Section 8.3: Quantum Policy Gradient (from scratch)

Policy gradient is the classical algorithm for training a policy directly:

1. Run the policy many times, collect rewards.
2. Compute a gradient that increases the probability of high-reward actions.
3. Update the policy parameters.

We can plug a quantum policy into that exact loop. The only quantum specific piece is computing the gradient of the circuit output with respect to its rotation angles, which we do with finite differences here.

In [ ]:
# Toy: 2-armed bandit. State is always 0, actions 0 and 1 give rewards drawn from
# different Gaussians. We train the 1-qubit quantum policy to prefer the better arm.
np.random.seed(0)
true_means = [0.2, 0.8]

def sample_reward(action):
    return np.random.normal(true_means[action], 0.1)

def numeric_grad(f, x, eps=1e-3):
    return (f(x+eps) - f(x-eps)) / (2*eps)

theta = 0.0
lr = 0.5
history = {'theta': [], 'avg_reward': []}
for step in range(30):
    # Sample 20 rollouts under current policy
    rewards = []
    for _ in range(20):
        p0, p1 = quantum_policy(0.0, theta)
        a = 0 if np.random.rand() < p0 else 1
        rewards.append(sample_reward(a))
    avg_r = np.mean(rewards)
    history['theta'].append(theta); history['avg_reward'].append(avg_r)

    # Gradient of expected reward w.r.t. theta (finite diff on the theoretical expectation)
    def exp_reward(t):
        p0, p1 = quantum_policy(0.0, t)
        return p0 * true_means[0] + p1 * true_means[1]
    g = numeric_grad(exp_reward, theta)
    theta = theta + lr * g   # gradient ASCENT

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history['theta'], color='#5B2C91', lw=2)
axes[0].set_xlabel('training step'); axes[0].set_ylabel('theta'); axes[0].set_title('Policy parameter')
axes[1].plot(history['avg_reward'], color='#2A9D8F', lw=2)
axes[1].axhline(max(true_means), color='gray', ls='--', label=f'best arm mean = {max(true_means)}')
axes[1].set_xlabel('training step'); axes[1].set_ylabel('avg reward'); axes[1].set_title('Learning curve')
axes[1].legend()
plt.tight_layout(); plt.show()

p0, p1 = quantum_policy(0.0, theta)
print(f"Final policy: P(action 0) = {p0:.3f}, P(action 1) = {p1:.3f}")
print(f"Best action was 1 (mean {true_means[1]}). Policy should heavily prefer action 1.")

---
# Section 8.4: Grover for Action Selection

In games with huge action spaces (like Go with roughly 300 legal moves at each turn, or continuous control with unbounded actions), evaluating every action is expensive.

Grover's algorithm (Unit 2) can help. If you have a black box that says "is this action good enough," Grover finds a good action in O(sqrt(N)) queries instead of O(N).

That is a *quadratic* speedup for action search, which for very large action spaces can be meaningful.

## Activity 8.4: Quantum vs classical action search

Below we compare the expected number of oracle calls needed to find a "good" action in an action space of size N, where a fraction f of actions are good.

In [ ]:
Ns = np.logspace(1, 6, 25).astype(int)
fraction_good = 0.01  # 1 percent of actions are "good"

classical_expected = 1 / fraction_good * np.ones_like(Ns, dtype=float)  # geometric random search
grover_expected    = np.sqrt(Ns / (fraction_good * Ns))                  # sqrt(N/good_count)

plt.loglog(Ns, classical_expected, marker='o', color='#E76F51', label='classical random search')
plt.loglog(Ns, grover_expected,    marker='o', color='#2A9D8F', label='Grover action search')
plt.xlabel('action space size N (log)')
plt.ylabel('expected oracle calls (log)')
plt.title(f'Finding a good action when {fraction_good*100:.0f}% of actions are good')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()
print("Classical needs a constant number of calls, Grover needs sqrt(N/k). Grover wins when N is huge.")

---
# End-of-Unit Quiz

Ten multiple choice questions. Reveal the answer to check yourself.

**1. In reinforcement learning, what is the goal of the agent?**

A) Predict labels for data
B) Compress data
C) Maximize cumulative future reward
D) Cluster similar states together

<details><summary>Answer 1</summary>**C).** RL agents choose actions to maximize the sum of (discounted) future rewards.</details>

---

**2. Which classical AI system uses reinforcement learning as a core component?**

A) MNIST digit classification
B) K-means clustering
C) AlphaGo
D) Linear regression on housing prices

<details><summary>Answer 2</summary>**C) AlphaGo.** Its self-play training is pure RL.</details>

---

**3. A quantum policy in RL is typically implemented as:**

A) A classical neural network with quantum weights
B) A parameterized quantum circuit whose measurement outcomes are actions
C) A quantum error correcting code
D) A quantum Fourier transform

<details><summary>Answer 3</summary>**B).** The circuit's measurement distribution IS the action distribution.</details>

---

**4. Grover's algorithm applied to RL action selection gives what kind of speedup?**

A) Exponential in the action space size
B) Quadratic in the action space size
C) Constant factor
D) No speedup, only a slowdown

<details><summary>Answer 4</summary>**B) quadratic.** O(N) becomes O(sqrt(N)) oracle calls.</details>

---

**5. Which RL bottleneck is quantum LEAST likely to help with on near term hardware?**

A) Estimating value functions over billions of states
B) Sampling from a stochastic policy
C) Running a giant classical simulator to train a robot
D) Compact policy representation

<details><summary>Answer 5</summary>**C).** Simulating a robot classically is not a quantum problem; it is a bottleneck of the environment, not the learning algorithm.</details>

---

**6. Which environment is SIMPLEST for teaching RL basics?**

A) A real physical robot
B) A 5 state chain (GridWorld style)
C) Full 3D robot arm simulation
D) Trading on real markets

<details><summary>Answer 6</summary>**B).** Small discrete environments like a chain or grid make the learning loop visible without heavy simulation cost.</details>

---

**7. What does the value function V(s) represent?**

A) The gradient of the policy
B) The expected cumulative future reward starting from state s
C) The one-step reward at state s
D) A fixed constant per state

<details><summary>Answer 7</summary>**B).** V(s) sums all future rewards you expect if you start in s and follow the current policy.</details>

---

**8. In RL, the tension between trying new actions and exploiting known good actions is called:**

A) The training vs test split
B) Exploration versus exploitation
C) Bias vs variance
D) Overfitting

<details><summary>Answer 8</summary>**B) Exploration vs exploitation.** Every RL agent has to balance the two.</details>

---

**9. Which technique combines RL with tree search and beat human champions at Go?**

A) K-means
B) Policy gradient plus Monte Carlo Tree Search (AlphaGo style)
C) Linear regression
D) Support vector machines

<details><summary>Answer 9</summary>**B).** AlphaGo combined a policy network, a value network, and MCTS.</details>

---

**10. Why might a compact policy representation be especially valuable in RL?**

A) It saves training time and reduces overfitting to noisy rollouts
B) It looks nicer in plots
C) It makes rewards larger
D) It has no benefit

<details><summary>Answer 10</summary>**A).** Fewer parameters generally means faster training and better generalization from limited episodes, which is exactly what RL usually has.</details>

---

## Unit 8 Summary

| Concept | Key idea |
|---|---|
| Reinforcement learning | Agent picks actions to maximize cumulative future reward. |
| Policy | Function from states to actions. |
| Value function | Expected future reward from a given state. |
| Quantum policy | Parameterized quantum circuit whose measurement outcomes are actions. |
| Quantum policy gradient | Classical policy gradient applied to a quantum policy's parameters. |
| Grover for action selection | Quadratic speedup in searching a large action space. |
| Applications | Games, robotics, LLMs, trading, recommenders. |

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 8!")